# 004 · XML with ElementTree

> **Standard Library › Data Formats** · Python 3.14

XML is still everywhere in enterprise integration: config files, SOAP payloads, RSS feeds, office documents. `xml.etree.ElementTree` is Python's lightweight, Pythonic XML API for parsing, querying and building documents.

**In this notebook you will learn:**

- how `ElementTree` maps to LINQ to XML (`XDocument` / `XElement`)
- parsing from strings and files, and navigating with `find`, `findall`, `iter` and `findtext`
- the supported XPath subset, and reading attributes
- building documents with `Element` / `SubElement` and serialising with `tostring` + `ET.indent`
- handling namespaces
- why untrusted XML should go through `defusedxml`

## .NET mental model

| LINQ to XML (`System.Xml.Linq`) | `xml.etree.ElementTree` (imported as `ET`) |
|---|---|
| `XDocument` | `ET.ElementTree` (a thin wrapper around the root) |
| `XElement` | `ET.Element` |
| `XElement.Parse(text)` | `ET.fromstring(text)` → root `Element` |
| `XDocument.Load(path)` | `ET.parse(path)` → `ElementTree` (`.getroot()`) |
| `el.Element("name")` | `el.find("name")` |
| `el.Elements("item")` | `el.findall("item")` (direct children) |
| `el.Descendants("item")` | `el.iter("item")` or `el.findall(".//item")` |
| `(string)el.Element("name")` | `el.findtext("name")` |
| `el.Attribute("id")?.Value` | `el.get("id")` |
| `el.Value` | `"".join(el.itertext())` — note `el.text` is only the text *before the first child* |
| `new XElement("a", new XElement("b"))` | `ET.Element("a")` + `ET.SubElement(a, "b")` |
| `XNamespace ns = "..."; ns + "item"` | `"{uri}item"` (Clark notation) |
| `XmlException` | `ET.ParseError` |

`System.Xml.XmlDocument` (the DOM) corresponds to `xml.dom.minidom`, which also exists in the standard library — but `ElementTree` is what you'll use in practice.

## Parsing from a string

`ET.fromstring` returns the root `Element`. An element has a `tag`, an `attrib` dict, `text`, and its children — and it is iterable like a list.

```csharp
// C#
var root = XElement.Parse(xml);
foreach (var product in root.Elements("product"))
    Console.WriteLine($"{product.Attribute("id")?.Value}: {(string)product.Element("name")}");
```

In [ ]:
import xml.etree.ElementTree as ET

catalog_xml = '''<catalog>
  <product id="1" category="fruit">
    <name>Apple</name>
    <price currency="EUR">0.50</price>
  </product>
  <product id="2" category="fruit">
    <name>Pear</name>
    <price currency="EUR">0.75</price>
  </product>
  <product id="3" category="tools">
    <name>Hammer</name>
    <price currency="USD">12.00</price>
    <tags><tag>steel</tag><tag>heavy</tag></tags>
  </product>
</catalog>'''

root = ET.fromstring(catalog_xml)
print(root.tag, len(root))           # tag name, number of direct children

for product in root:                 # iterate direct children
    print(product.tag, product.attrib)

## Navigating: `find`, `findall`, `findtext`, `iter`

- `find(path)` — first matching element or `None` (like `Element(...)`)
- `findall(path)` — list of matches (like `Elements(...)`)
- `findtext(path, default=None)` — text of the first match (like the `(string)` cast)
- `iter(tag)` — every descendant with that tag, at any depth (like `Descendants(...)`)

In [ ]:
first = root.find("product")
print(first.get("id"), first.find("name").text)

print([p.findtext("name") for p in root.findall("product")])

print(root.findtext("product/price"))            # path through children
print(root.findtext("product/discount", "none")) # default when missing

print([tag.text for tag in root.iter("tag")])    # any depth

⚠️ **Gotcha:** `find` returns `None` when nothing matches, so `root.find("missing").text` raises `AttributeError`. Prefer `findtext(path, default)`, or check for `None` first.

In [ ]:
try:
    root.find("product/discount").text
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** an `Element` with **no children** is *falsy*, so never write `if element:` to test whether `find` succeeded — write `if element is not None:`.

In [ ]:
name = root.find("product/name")
print(name is not None, len(name), bool(len(name)))   # found, but it has no children

## Attributes

`.attrib` is a plain `dict`; `.get(name, default)` reads one attribute safely; `.set(name, value)` writes. Attribute values are always strings.

In [ ]:
hammer = root.find("product[@id='3']")
print(hammer.attrib)
print(hammer.get("category"), hammer.get("stock", "unknown"))

price = hammer.find("price")
print(price.get("currency"), float(price.text))   # convert text yourself

## The XPath subset

`ElementTree` supports a **limited** XPath — enough for most queries. For full XPath 1.0 you'd use the third-party `lxml` package (comparable to `System.Xml.XPath`).

| Syntax | Meaning |
|---|---|
| `tag`, `*`, `.`, `..` | child by name, any child, self, parent |
| `.//tag` | descendants at any depth |
| `[@attr]`, `[@attr='v']` | has attribute / attribute equals |
| `[tag]`, `[tag='text']` | has child / child's text equals |
| `[1]`, `[last()]`, `[last()-1]` | position (1-based) |

In [ ]:
print([p.get("id") for p in root.findall("product[@category='fruit']")])
print(root.find("product[name='Pear']").get("id"))
print(root.findtext("product[last()]/name"))
print([p.get("id") for p in root.findall(".//price[@currency='USD']/..")])
print([t.text for t in root.findall(".//tags/tag")])

There is no `text()`, no `contains()`, no arithmetic, and paths can't start with `/`. Use Python for anything beyond the subset — a comprehension over `iter()` is often clearer anyway:

In [ ]:
try:
    root.findall("//product")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")

cheap = [p.findtext("name") for p in root.iter("product") if float(p.findtext("price")) < 1]
print(cheap)

## Reading from a file

`ET.parse` accepts a path or a binary file object and returns an `ElementTree`; call `.getroot()` for the root element. The XML declaration decides the encoding, so you don't pass one.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "catalog.xml"
    path.write_text(catalog_xml, encoding="utf-8")

    tree = ET.parse(path)
    file_root = tree.getroot()
    print(type(tree).__name__, file_root.tag, len(file_root.findall("product")))

**Tip:** for very large files, `ET.iterparse(path, events=("end",))` streams elements one by one (like `XmlReader`), so you can process and `clear()` them without holding the whole document in memory.

## Building a document

```csharp
// C#
var order = new XElement("order", new XAttribute("id", 1001),
    new XElement("customer", "alice"),
    new XElement("items",
        new XElement("item", new XAttribute("sku", "A1"), new XAttribute("qty", 2))));
```

In Python, create the root with `Element` and add children with `SubElement(parent, tag, attrib)`. Attribute values and text must be **strings**.

In [ ]:
order = ET.Element("order", id="1001")
ET.SubElement(order, "customer").text = "alice"

items = ET.SubElement(order, "items")
for sku, qty in [("A1", 2), ("B7", 1)]:
    ET.SubElement(items, "item", sku=sku, qty=str(qty))

print(ET.tostring(order, encoding="unicode"))

## Serialising: `tostring`, `ET.indent`, `write`

- `ET.tostring(el)` returns **bytes** by default; pass `encoding="unicode"` for a `str`.
- `ET.indent(tree_or_element, space="  ")` (Python 3.9+) pretty-prints **in place**. (`XElement.ToString()` indents by default; `ET.tostring` never does.)
- `ElementTree(root).write(path, encoding="utf-8", xml_declaration=True)` writes a file.

In [ ]:
print(type(ET.tostring(order)).__name__)     # bytes by default

ET.indent(order, space="  ")
print(ET.tostring(order, encoding="unicode"))

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "order.xml"
    ET.ElementTree(order).write(path, encoding="utf-8", xml_declaration=True)
    print(path.read_text(encoding="utf-8"))

Modifying an existing tree works the same way: change `.text`, call `.set()`, `append()` / `remove()` children.

In [ ]:
for item in order.iter("item"):
    item.set("qty", str(int(item.get("qty")) * 10))
order.find("items").remove(order.find("items/item[@sku='B7']"))
ET.indent(order)     # re-indent: the old whitespace (.tail) was left behind
print(ET.tostring(order, encoding="unicode"))

## Namespaces

ElementTree stores namespaced names in *Clark notation*: `{uri}local`. To query, either write that form, pass a `namespaces` prefix map, or use the `{*}` wildcard (any namespace).

```csharp
// C#
XNamespace c = "urn:example:catalog";
var names = root.Elements(c + "product").Select(p => (string)p.Element(c + "name"));
```

In [ ]:
ns_xml = '''<catalog xmlns="urn:example:catalog" xmlns:p="urn:example:pricing">
  <product id="1"><name>Apple</name><p:price currency="EUR">0.50</p:price></product>
</catalog>'''

ns_root = ET.fromstring(ns_xml)
print(ns_root.tag)                                    # Clark notation

ns = {"c": "urn:example:catalog", "p": "urn:example:pricing"}
print(ns_root.findtext("c:product/c:name", namespaces=ns))
print(ns_root.findtext("{urn:example:catalog}product/{urn:example:pricing}price"))
print(ns_root.findtext("{*}product/{*}price"))         # wildcard

print(ns_root.find("c:product", ns).find("c:name", ns) is not None)
print(ns_root.find("product"))                        # None — the default namespace applies!

⚠️ **Gotcha:** a default namespace (`xmlns="..."`) applies to every unprefixed element, so plain `find("product")` finds nothing — the same trap as forgetting the `XNamespace` in LINQ to XML.

When serialising, ElementTree invents prefixes like `ns0:` unless you call `ET.register_namespace(prefix, uri)` — note that this registration is **global** for the process. (`.strip()` below just removes the element's trailing whitespace, its `.tail`, which `tostring` includes.)

In [ ]:
print(ET.tostring(ns_root.find("{*}product"), encoding="unicode").strip())

ET.register_namespace("", "urn:example:catalog")
ET.register_namespace("p", "urn:example:pricing")
print(ET.tostring(ns_root.find("{*}product"), encoding="unicode").strip())

## Errors

Malformed XML raises `ET.ParseError` (a `SyntaxError` subclass) with a `.position` tuple of `(line, column)`.

In [ ]:
try:
    ET.fromstring("<order><item></order>")
except ET.ParseError as e:
    print(f"{type(e).__name__}: {e}")
    print(e.position)

## ⚠️ Security: untrusted XML

XML has features that can be abused — entity expansion ("billion laughs") and, in some parsers, external entities (XXE). .NET hardened its defaults (`DtdProcessing.Prohibit` in `XmlReaderSettings`); the Python documentation explicitly warns that the `xml` modules are **not** secure against maliciously constructed data.

For XML from users, uploads or third-party systems, use the **`defusedxml`** package (third-party, not in the standard library). It is a drop-in replacement:

```python
# pip install defusedxml
import defusedxml.ElementTree as DET
root = DET.fromstring(untrusted_text)   # same API, refuses dangerous constructs
```

For XML you produced yourself (config files, test fixtures), the standard library is fine.

## Summary

| Task | C# (LINQ to XML) | Python (`ElementTree`) |
|---|---|---|
| Parse a string | `XElement.Parse(s)` | `ET.fromstring(s)` |
| Load a file | `XDocument.Load(path)` | `ET.parse(path).getroot()` |
| First child | `el.Element("x")` | `el.find("x")` |
| All children | `el.Elements("x")` | `el.findall("x")` |
| All descendants | `el.Descendants("x")` | `el.iter("x")` / `findall(".//x")` |
| Child text | `(string)el.Element("x")` | `el.findtext("x", default)` |
| Attribute | `el.Attribute("a")?.Value` | `el.get("a")` |
| Build | `new XElement("a", ...)` | `ET.Element("a")`, `ET.SubElement(p, "b")` |
| To string | `el.ToString()` | `ET.tostring(el, encoding="unicode")` |
| Pretty-print | default `ToString()` formatting | `ET.indent(el)` |
| Namespaces | `XNamespace ns; ns + "x"` | `"{uri}x"` or `find("p:x", namespaces)` |
| Parse error | `XmlException` | `ET.ParseError` |

## Exercises

1. Using `catalog_xml` from this notebook, build a `dict` mapping product name → price as `float`, only for products priced in `EUR`.
2. Build this document with `Element`/`SubElement`, pretty-print it and print it as a string:
   `<sensors><sensor id="t1" unit="C">21.5</sensor><sensor id="t2" unit="C">19.0</sensor></sensors>`
3. Parse `<feed xmlns="http://www.w3.org/2005/Atom"><entry><title>One</title></entry><entry><title>Two</title></entry></feed>` and print all entry titles — once with a `namespaces` map and once with the `{*}` wildcard.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import xml.etree.ElementTree as ET

catalog_xml = '''<catalog>
  <product id="1" category="fruit"><name>Apple</name><price currency="EUR">0.50</price></product>
  <product id="2" category="fruit"><name>Pear</name><price currency="EUR">0.75</price></product>
  <product id="3" category="tools"><name>Hammer</name><price currency="USD">12.00</price></product>
</catalog>'''

# 1 — filter on the price element's currency attribute
root = ET.fromstring(catalog_xml)
eur = {p.findtext("name"): float(p.findtext("price"))
       for p in root.findall("product[price]")
       if p.find("price").get("currency") == "EUR"}
print(eur)   # {'Apple': 0.5, 'Pear': 0.75}

# 2 — text and attribute values must be strings
sensors = ET.Element("sensors")
for sensor_id, value in [("t1", 21.5), ("t2", 19.0)]:
    ET.SubElement(sensors, "sensor", id=sensor_id, unit="C").text = str(value)
ET.indent(sensors)
print(ET.tostring(sensors, encoding="unicode"))

# 3 — the default namespace applies to every element
feed = ET.fromstring('<feed xmlns="http://www.w3.org/2005/Atom"><entry><title>One</title></entry>'
                     '<entry><title>Two</title></entry></feed>')
ns = {"a": "http://www.w3.org/2005/Atom"}
print([e.findtext("a:title", namespaces=ns) for e in feed.findall("a:entry", ns)])
print([t.text for t in feed.findall("{*}entry/{*}title")])
```

</details>